# VERA Phase 2 - Quick Start Tutorial

This notebook demonstrates the basic usage of the VERA Phase 2 system for diabetic retinopathy detection.

## What You'll Learn

1. Loading and preprocessing fundus images
2. Running vessel segmentation
3. Making DR predictions
4. Generating explainability visualizations

## Prerequisites

Make sure you have:
- Installed all requirements: `pip install -r requirements.txt`
- Downloaded at least one dataset (APTOS, EyePACS, or Messidor-2)
- Trained models (or download pretrained checkpoints)

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.data.preprocessing import FundusPreprocessor
from src.models.vessel_segmentation import UNetVesselSegmenter
from src.models.classification import DRClassifier
from src.explainability.gradcam import GradCAMPlusPlus
from src.explainability.overlap_score import compute_vessel_attention_overlap
from src.utils.config import load_config

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## Step 1: Load Configuration and Models

In [ ]:
# Load configuration
config = load_config('../configs/config.yaml')

# Load vessel segmentation model
vessel_segmenter = UNetVesselSegmenter(
    encoder_name=config['vessel_segmentation']['encoder'],
    pretrained=False
).to(device)

vessel_checkpoint = torch.load('../models/checkpoints/vessel_segmenter.pth', map_location=device)
vessel_segmenter.load_state_dict(vessel_checkpoint['model_state_dict'])
vessel_segmenter.eval()

print("✓ Vessel segmenter loaded")

# Load DR classifier
dr_model = DRClassifier(
    backbone_name=config['model']['backbone'],
    fusion_strategy=config['model']['fusion_strategy'],
    num_classes=5,
    pretrained=False
).to(device)

dr_checkpoint = torch.load('../models/checkpoints/best_model.pth', map_location=device)
dr_model.load_state_dict(dr_checkpoint['model_state_dict'])
dr_model.eval()

print("✓ DR classifier loaded")
print(f"  - Backbone: {config['model']['backbone']}")
print(f"  - Fusion: {config['model']['fusion_strategy']}")
print(f"  - Validation QWK: {dr_checkpoint.get('val_qwk', 'N/A')}")

## Step 2: Load and Preprocess an Image

In [ ]:
# Load a sample fundus image
image_path = '../data/sample_images/example_fundus.png'  # Replace with your image
image_bgr = cv2.imread(image_path)

if image_bgr is None:
    raise FileNotFoundError(f"Image not found: {image_path}")

# Display original
plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
plt.title('Original Fundus Image')
plt.axis('off')
plt.show()

# Create preprocessor
preprocessor = FundusPreprocessor(
    target_size=(config['data']['image_size'], config['data']['image_size']),
    circular_crop=True,
    ben_graham_enabled=True,
    clahe_enabled=True
)

# Preprocess
processed_image = preprocessor.preprocess(image_bgr)

# Display preprocessed
plt.figure(figsize=(6, 6))
plt.imshow(processed_image)
plt.title('Preprocessed Image (Ben Graham + CLAHE)')
plt.axis('off')
plt.show()

# Convert to tensor
image_tensor = torch.from_numpy(processed_image).permute(2, 0, 1).float().unsqueeze(0).to(device)
print(f"Image tensor shape: {image_tensor.shape}")

## Step 3: Segment Blood Vessels

In [ ]:
# Segment vessels
with torch.no_grad():
    vessel_logits = vessel_segmenter(image_tensor)
    vessel_probs = torch.sigmoid(vessel_logits)
    vessel_binary = (vessel_probs > 0.5).float()

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(processed_image)
axes[0].set_title('Preprocessed Fundus')
axes[0].axis('off')

axes[1].imshow(vessel_probs.squeeze().cpu().numpy(), cmap='gray')
axes[1].set_title('Vessel Probability Map')
axes[1].axis('off')

axes[2].imshow(vessel_binary.squeeze().cpu().numpy(), cmap='gray')
axes[2].set_title('Binary Vessel Segmentation')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print(f"Vessel tensor shape: {vessel_binary.shape}")
print(f"Vessel coverage: {vessel_binary.mean().item():.2%}")

## Step 4: Predict DR Grade

In [ ]:
# Run inference
with torch.no_grad():
    logits = dr_model(image_tensor, vessel_binary)
    probabilities = torch.softmax(logits, dim=1)
    prediction = logits.argmax(dim=1).item()
    confidence = probabilities[0, prediction].item()

# Display results
grade_names = ['No DR', 'Mild NPDR', 'Moderate NPDR', 'Severe NPDR', 'Proliferative DR']
grade_colors = ['green', 'blue', 'orange', 'darkorange', 'red']

print(f"\n{'='*50}")
print(f"PREDICTION: Grade {prediction} - {grade_names[prediction]}")
print(f"CONFIDENCE: {confidence:.2%}")
print(f"{'='*50}\n")

# Plot probability distribution
plt.figure(figsize=(10, 5))
bars = plt.bar(grade_names, probabilities.squeeze().cpu().numpy(), 
               color=grade_colors, alpha=0.7, edgecolor='black')
bars[prediction].set_alpha(1.0)
bars[prediction].set_edgecolor('red')
bars[prediction].set_linewidth(3)

plt.ylabel('Probability', fontweight='bold', fontsize=12)
plt.title('DR Grade Probabilities', fontweight='bold', fontsize=14)
plt.ylim([0, 1])
plt.grid(axis='y', alpha=0.3)

# Add value labels
for bar, prob in zip(bars, probabilities.squeeze().cpu().numpy()):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{prob:.1%}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

## Step 5: Generate Explainability with Grad-CAM++

In [ ]:
# Get target layer (last conv layer of backbone)
if hasattr(dr_model.backbone, 'model'):
    if hasattr(dr_model.backbone.model, 'layer4'):
        target_layer = dr_model.backbone.model.layer4[-1]  # ResNet
    else:
        target_layer = list(dr_model.backbone.model.children())[-2]  # Generic
else:
    target_layer = list(dr_model.backbone.children())[-2]

# Create Grad-CAM++
gradcam = GradCAMPlusPlus(dr_model.backbone, target_layer)

# Generate heatmap
heatmap = gradcam.generate_heatmap(image_tensor, vessel_binary, target_class=prediction)

# Create colored heatmap
heatmap_colored = cv2.applyColorMap((heatmap * 255).astype(np.uint8), cv2.COLORMAP_JET)
overlay = cv2.addWeighted(
    cv2.resize(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB), (heatmap.shape[1], heatmap.shape[0])),
    0.6,
    heatmap_colored,
    0.4,
    0
)

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(processed_image)
axes[0].set_title('Original Image', fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(heatmap, cmap='jet')
axes[1].set_title('Attention Heatmap (Grad-CAM++)', fontsize=14, fontweight='bold')
axes[1].axis('off')

axes[2].imshow(overlay)
axes[2].set_title('Overlay on Fundus', fontsize=14, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("\nInterpretation: Red areas show where the model focused to make its prediction.")

## Step 6: Compute Vessel-Attention Overlap

In [ ]:
# Compute overlap metrics
overlap_metrics = compute_vessel_attention_overlap(
    heatmap,
    vessel_binary.squeeze().cpu().numpy()
)

print("\nVessel-Attention Overlap Metrics:")
print(f"  Overlap Score:      {overlap_metrics['overlap_score']:.3f}")
print(f"  Vessel Coverage:    {overlap_metrics['coverage']:.3f}")
print(f"  Attention Precision: {overlap_metrics['precision']:.3f}")
print(f"  Vessel Area:        {overlap_metrics['vessel_area']:.3f}")
print(f"  Attention Area:     {overlap_metrics['attention_area']:.3f}")

# Visualize overlap
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(vessel_binary.squeeze().cpu().numpy(), cmap='gray')
axes[0].set_title('Vessel Segmentation', fontsize=14, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(heatmap > 0.5, cmap='hot')
axes[1].set_title('High Attention Regions', fontsize=14, fontweight='bold')
axes[1].axis('off')

# Create overlap visualization
vessel_np = vessel_binary.squeeze().cpu().numpy()
attention_binary = heatmap > 0.5
overlap_vis = np.zeros((*heatmap.shape, 3))
overlap_vis[vessel_np > 0.5] = [0, 1, 0]  # Green = vessels
overlap_vis[attention_binary] = [1, 0, 0]  # Red = attention
overlap_vis[(vessel_np > 0.5) & attention_binary] = [1, 1, 0]  # Yellow = overlap

axes[2].imshow(overlap_vis)
axes[2].set_title('Overlap (Yellow = Vessel + Attention)', fontsize=14, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

print("\n✓ Analysis complete!")

## Summary

In this notebook, you learned how to:

1. ✓ Load and preprocess fundus images with Ben Graham and CLAHE
2. ✓ Segment blood vessels using U-Net
3. ✓ Predict DR severity grades with vessel-aware fusion models
4. ✓ Generate explainability visualizations with Grad-CAM++
5. ✓ Quantify vessel-attention alignment

## Next Steps

- Try `02_training_tutorial.ipynb` to learn how to train your own models
- Explore `03_ablation_studies.ipynb` for systematic experiments
- See `04_evaluation_deep_dive.ipynb` for comprehensive evaluation techniques

## Additional Resources

- **API Reference:** `../API_REFERENCE.md`
- **Usage Guide:** `../USAGE_GUIDE.md`
- **Configuration:** `../configs/config.yaml`